<a href="https://colab.research.google.com/github/khrixtar-dev/Proyecto_Movilidad_RomeroOssesSagredoParedes/blob/rama_cris/ProyectoMovilidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import os
from google.colab import drive

# 1. Montar Google Drive
# Esto abrirá una ventana emergente para pedir permisos de acceso
drive.mount('/content/drive')

# 2. Definir Rutas (Basadas en tu carpeta compartida)
# Nota: 'MyDrive' es la carpeta raíz de tu Google Drive
path_raiz = '/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/TallerPractico_1/'
path_raw = path_raiz + 'data/raw/Global_Mobility_Report.zip'
path_processed = path_raiz + 'data/processed/'

# Crear la carpeta 'processed' si los alumnos no la han creado aún
if not os.path.exists(path_processed):
    os.makedirs(path_processed)
    print(f"Carpeta creada en: {path_processed}")

# 3. Procesamiento por Trozos (Chunking)
# Leemos de a 100.000 filas para no colapsar la memoria RAM
pedazos_chile = []

print("Iniciando lectura y filtrado de datos (esto puede tardar un par de minutos)...")

try:
    # Leemos el ZIP directamente sin descomprimir en el disco
    for chunk in pd.read_csv(path_raw, compression='zip', chunksize=100000, low_memory=False):
        # Filtramos inmediatamente cada trozo por el país 'Chile'
        filtrado = chunk[chunk['country_region'] == 'Chile']

        # Solo guardamos el pedazo si encontró datos de Chile
        if not filtrado.empty:
            pedazos_chile.append(filtrado)

    # 4. Consolidación y Salida
    if pedazos_chile:
        df_chile = pd.concat(pedazos_chile, ignore_index=True)

        # Guardamos el resultado "limpio" en un CSV mucho más pequeño
        archivo_salida = os.path.join(path_processed, 'Chile_Mobility_Clean.csv')
        df_chile.to_csv(archivo_salida, index=False)

        print("-" * 50)
        print(f"¡PROCESO COMPLETADO!")
        print(f"Archivo generado: {archivo_salida}")
        print(f"Registros totales de Chile: {len(df_chile)}")
        print("-" * 50)
    else:
        print("No se encontraron datos para 'Chile' en el archivo.")

except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en {path_raw}")
    print("Asegúrate de que la carpeta 'data/raw' existe y el ZIP tiene el nombre correcto.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Iniciando lectura y filtrado de datos (esto puede tardar un par de minutos)...
--------------------------------------------------
¡PROCESO COMPLETADO!
Archivo generado: /content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/TallerPractico_1/data/processed/Chile_Mobility_Clean.csv
Registros totales de Chile: 68716
--------------------------------------------------


In [17]:
df_chile.head()

,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,2.0,4.0,9.0,0.0,-3.0,0.0
1,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-16,3.0,5.0,5.0,4.0,-1.0,0.0
2,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-17,1.0,6.0,11.0,-3.0,-8.0,1.0
3,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-18,0.0,5.0,13.0,-3.0,-7.0,1.0
4,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-19,0.0,8.0,11.0,-3.0,-7.0,1.0


##Fase 2 Limpieza de datos

In [30]:
df_chile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68716 entries, 0 to 68715
Data columns (total 14 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   country_region_code                                 68716 non-null  object        
 1   country_region                                      68716 non-null  object        
 2   sub_region_1                                        67742 non-null  object        
 3   sub_region_2                                        52160 non-null  object        
 4   iso_3166_2_code                                     15582 non-null  object        
 5   place_id                                            68716 non-null  object        
 6   date                                                68716 non-null  datetime64[ns]
 7   retail_and_recreation_percent_change_from_baseline  62421 non-null  float64       
 8   grocer

###Nulos

In [27]:
#Cuantos nulos hay por serie
total_registros = df_chile['country_region_code'].count()
print(f'Cantidad de registros: {total_registros}')

#Normal
df_chile.isna().sum()
#Porcentaje
round(df_chile.isna().sum()*100/total_registros, 2)



Cantidad de registros: 68716


,0
country_region_code,0.00
country_region,0.00
sub_region_1,1.42
sub_region_2,24.09
iso_3166_2_code,77.32
place_id,0.00
date,0.00
retail_and_recreation_percent_change_from_baseline,9.16
grocery_and_pharmacy_percent_change_from_baseline,15.31
parks_percent_change_from_baseline,0.27


In [20]:
#Borrar columnas completamente nulas
df_chile = df_chile.drop(columns=['metro_area', 'census_fips_code'])

In [21]:
#Nulos en variables numericas
#Se rellenaran con valores promedio pero antes se detectaran outliers

###Duplicados

In [22]:
#Duplicados por grupo
conteo = df_chile.groupby(['date', 'sub_region_1']).size().reset_index(name='count')
duplicados_1 = conteo[conteo['count'] > 1]
duplicados_1

,date,sub_region_1,count
0,2020-02-15,Antofagasta,4
1,2020-02-15,Araucania,3
2,2020-02-15,Arica y Parinacota,2
3,2020-02-15,Atacama,4
4,2020-02-15,Aysén,4
...,...,...,...
15577,2022-10-15,O'Higgins,4
15578,2022-10-15,Santiago Metropolitan Region,7
15579,2022-10-15,Tarapacá,3
15580,2022-10-15,Valparaíso,9


In [23]:
conteo2 = df_chile.groupby(['date', 'sub_region_2']).size().reset_index(name='count')
duplicados_2 = conteo2[conteo2['count'] > 1]
duplicados_2

,date,sub_region_2,count


###Outliers

In [ ]:
#     retail_and_recreation_percent_change_from_baseline
#     grocery_and_pharmacy_percent_change_from_baseline
#     parks_percent_change_from_baseline
#     transit_stations_percent_change_from_baseline
#     workplaces_percent_change_from_baseline
#     residential_percent_change_from_baseline

In [24]:
#Variable para cambiar
col = 'retail_and_recreation_percent_change_from_baseline'

df_var = df_chile[['country_region', 'sub_region_1', 'sub_region_2', col]].copy()

#Eliminar nulos, solo de la variable
df_var = df_var.dropna(subset=[col])

Q1 = df_var[col].quantile(0.25)
Q3 = df_var[col].quantile(0.75)
IQR = Q3 - Q1


lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df_var[(df_var[col] < lower) | (df_var[col] > upper)]

print(f"Cantidad de outliers en {col}:", len(outliers))
outliers.head()

Cantidad de outliers en retail_and_recreation_percent_change_from_baseline: 58


,country_region,sub_region_1,sub_region_2,retail_and_recreation_percent_change_from_baseline
39380,Chile,Maule,Cauquenes Province,74.0
39381,Chile,Maule,Cauquenes Province,73.0
39382,Chile,Maule,Cauquenes Province,74.0
39383,Chile,Maule,Cauquenes Province,83.0
39386,Chile,Maule,Cauquenes Province,103.0


In [ ]:
##QUE HACEMOS CON LOS OUTLIEEEERSS?!?!?!?!?!?!?!?!?!

## FASE 3

In [25]:
#Tipado
df_chile['date'] = pd.to_datetime(df_chile['date'])

In [31]:
df_chile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68716 entries, 0 to 68715
Data columns (total 14 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   country_region_code                                 68716 non-null  object        
 1   country_region                                      68716 non-null  object        
 2   sub_region_1                                        67742 non-null  object        
 3   sub_region_2                                        52160 non-null  object        
 4   iso_3166_2_code                                     15582 non-null  object        
 5   place_id                                            68716 non-null  object        
 6   date                                                68716 non-null  datetime64[ns]
 7   retail_and_recreation_percent_change_from_baseline  62421 non-null  float64       
 8   grocer

In [28]:
#Ingenieria
df_chile['movilidad_promedio'] = df_chile.iloc[:, 7:13].mean(axis=1)

In [29]:
df_chile.head()

,country_region_code,country_region,sub_region_1,sub_region_2,iso_3166_2_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline,movilidad_promedio
0,CL,Chile,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,2.0,4.0,9.0,0.0,-3.0,0.0,2.000000
1,CL,Chile,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-16,3.0,5.0,5.0,4.0,-1.0,0.0,2.666667
2,CL,Chile,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-17,1.0,6.0,11.0,-3.0,-8.0,1.0,1.333333
3,CL,Chile,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-18,0.0,5.0,13.0,-3.0,-7.0,1.0,1.500000
4,CL,Chile,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-19,0.0,8.0,11.0,-3.0,-7.0,1.0,1.666667


In [33]:
#Escalamiento y estandarizacion
##Usar MinMaxScaler seria peligroso ya que es muy sensible a outliers, StandarScaler es menos sensible pero tiende a ser un problema igualmente, la mejor opcion es usar RobustScaler ya que usa la mediana y el IQR
